# Strands structured output — step by step

This notebook is the step-by-step version of [`structured_output.py`](structured_output.py).

Unlike the pydantic-only notebook, this one **does call an LLM**: `agent(...)`
sends a prompt to Amazon Bedrock and Strands returns a validated `PersonInfo`
object instead of free text.

**Requirements:** AWS credentials with access to `amazon.nova-lite-v1:0`
(the only model permitted in the class environment). Run the cells in order.

In [1]:
from pydantic import BaseModel, Field
from strands import Agent

## 1) Define the pydantic model — the contract

This is the *same* class as in `pydantic_only.ipynb`. Nothing about it is
Strands-specific: it is a plain pydantic model.

What changes is its *role*. Here it acts as the **contract** between your code
and the LLM: you tell Strands "the answer must have this shape", and Strands
guarantees you get back either an instance of this class or an exception —
never a string you have to parse.

Remember: the docstring and the `Field(description=...)` texts are what the
model reads. They are your only way to explain the fields to it.

In [2]:
class PersonInfo(BaseModel):
    """Model that contains information about a Person"""
    name: str = Field(description="Name of the person")
    age: int = Field(description="Age of the person")
    occupation: str = Field(description="Occupation of the person")

## 2) Create the agent

A minimal agent: just a model id. No tools, no system prompt.

Note what is **not** here: we do not tell the agent about `PersonInfo` yet. The
output model is passed *per call*, in the next step — the same agent can return
different shapes for different questions. (You can also set a default with
`Agent(structured_output_model=...)` if one agent always returns the same shape.)

In [3]:
agent = Agent(
    model="amazon.nova-lite-v1:0"  # the only model permitted in this environment
)

## 3) Call the agent with `structured_output_model=`

This is the one line that turns a chat agent into a structured-data extractor.

What happens inside Strands when this cell runs
(details with source references in [`README.md`](README.md)):

1. `PersonInfo.model_json_schema()` is converted into a **tool specification**
   named `PersonInfo`.
2. That tool is offered to the model, and the model is required to call it to
   finish its answer.
3. The model responds with a `toolUse` block: `PersonInfo(name=..., age=..., occupation=...)`.
4. Strands runs `PersonInfo(**arguments)` — pydantic coerces and validates.
5. If validation fails, the error text goes back to the model as a tool error
   and the model retries. If it succeeds, the instance is stored in
   `result.structured_output`.

Watch the cell output: you should see the tool call happening
(`Tool #1: PersonInfo`) before the final result.

In [4]:
# Spec: https://strandsagents.com/docs/api/python/strands.agent.agent/
result = agent(
    "John Smith is a 30 year-old software engineer",
    structured_output_model=PersonInfo
)

<thinking> The User has provided information about a person named John Smith. I need to store this information in the

 PersonInfo tool. </thinking>

Tool #1: PersonInfo


## 4) Use the result — typed fields, no parsing

`result.structured_output` **is a `PersonInfo` instance**. You access fields as
attributes, and `age` is already an `int` — no `json.loads`, no regex, no
`int(...)` conversion.

Compare this to the alternative: asking the model "reply in JSON" and hoping the
reply parses. Here the shape is *enforced*, not *requested*.

In [5]:
person_info: PersonInfo = result.structured_output

print(f"Name: {person_info.name}")       # "John Smith"
print(f"Age:  {person_info.age}")        # 30  (an int!)
print(f"Job:  {person_info.occupation}") # "software engineer"
print()
print(type(person_info), "| age is", type(person_info.age).__name__)

Name: John Smith
Age:  30
Job:  software engineer

<class '__main__.PersonInfo'> | age is int


## 5) Look under the hood: the message history

Everything above was "ordinary tool calling". We can prove it by inspecting the
agent's conversation history. Look for:

- an **assistant** message containing a `toolUse` block with `name: PersonInfo`
  and your three fields as `input`
- a **user** message containing the matching `toolResult`

That tool call *is* the structured output. Pydantic just ran on its arguments.

In [6]:
import json

for msg in agent.messages:
    for block in msg["content"]:
        if "toolUse" in block:
            print(f"[{msg['role']}] toolUse  name={block['toolUse']['name']}")
            print("           input =", json.dumps(block["toolUse"]["input"]))
        elif "toolResult" in block:
            print(f"[{msg['role']}] toolResult status={block['toolResult']['status']}")
        elif "text" in block and block["text"].strip():
            print(f"[{msg['role']}] text     {block['text'][:80]!r}")

[user] text     'John Smith is a 30 year-old software engineer'
[assistant] text     '<thinking> The User has provided information about a person named John Smith. I '
[assistant] toolUse  name=PersonInfo
           input = {"occupation": "software engineer", "name": "John Smith", "age": 30}
[user] toolResult status=success


## Try it yourself

- Change the prompt to *"Ada Lovelace, born 1815, wrote the first algorithm"* —
  there is no explicit age. What does the model put into `age`? (The schema
  guarantees an `int`, not that the `int` is *right*.)
- Add `age: int = Field(description=..., ge=0, le=120)` and give the model an
  absurd age in the prompt. Re-run step 5 and look for a `toolResult` with
  `status=error` followed by a second `toolUse` — that is the retry loop.
- Add a fourth field with a `Literal[...]` type and see how it appears as an
  `enum` in `PersonInfo.model_json_schema()`.

Next: [`../explain_structured_output.py`](../explain_structured_output.py) shows
the retry loop on the real CV-match schemas, and
[`../workflow.py`](../workflow.py) chains two structured-output agents into a
complete workflow.